In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Install Dependencies

In [ ]:
!pip install -q transformers datasets evaluate rouge-score bert-score sentencepiece accelerate

# Environment Setup & Data Loading
Loading the raw CSVs, cleans the column headers, and constructs the dual-task prompt pairs.

In [ ]:
import os
import gc
import re
import time
import torch
import pandas as pd
from datasets import Dataset
from tqdm.auto import tqdm
from transformers import (
    AutoTokenizer, 
    AutoModelForSeq2SeqLM, 
    Seq2SeqTrainer, 
    Seq2SeqTrainingArguments, 
    DataCollatorForSeq2Seq
)

# 1. Reset memory state
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# 2. Load dataset
COMP_DIR = '/kaggle/input/competitions/dsn-bootcamp-hackathon-2026-llm-agent-track'
raw_df  = pd.read_csv(os.path.join(COMP_DIR, 'train.csv')).rename(columns={'category': 'label', 'lang': 'language'})
test_df = pd.read_csv(os.path.join(COMP_DIR, 'test.csv')).rename(columns={'lang': 'language'})

train_df = raw_df[raw_df['split'] == 'train'].reset_index(drop=True)
dev_df   = raw_df[raw_df['split'] != 'train'].reset_index(drop=True)

# 3. Format dual-task pairs (Task A & Task B)
def prepare_dual_task_data(df, is_test=False):
    records = []
    for _, row in df.iterrows():
        lang = str(row['language']).strip()
        text = str(row['text']).strip()
        doc_id = str(row['id']).strip()
        
        # Task A: Topic
        input_topic = f"classify topic in {lang}: {text}"
        if not is_test:
            records.append({'id': f"{doc_id}_topic", 'input_text': input_topic, 'target_text': str(row['label']).strip().lower(), 'task': 'topic'})
        else:
            records.append({'id': f"{doc_id}_topic", 'input_text': input_topic, 'task': 'topic'})
            
        # Task B: Headline
        input_headline = f"generate headline in {lang}: {text}"
        if not is_test:
            records.append({'id': f"{doc_id}_headline", 'input_text': input_headline, 'target_text': str(row['headline']).strip(), 'task': 'headline'})
        else:
            records.append({'id': f"{doc_id}_headline", 'input_text': input_headline, 'task': 'headline'})
    return pd.DataFrame(records)

train_dual = prepare_dual_task_data(train_df)
test_dual  = prepare_dual_task_data(test_df, is_test=True)

print(f"Data ready: {len(train_dual)} train pairs, {len(test_dual)} test pairs.")

# Model Architecture & Parameter Verification
Load google/mt5-small and prints the parameter count required for competition grading.

In [ ]:
MODEL_NAME = "google/mt5-small"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)

total_params = sum(p.numel() for p in model.parameters())
print("=" * 60)
print(f"Model: {MODEL_NAME}")
print(f"Total Parameters: {total_params:,} (Compliant with < 1B limit)")
print("=" * 60)

# Tokenization
Convert the text prompts and labels into token IDs with fixed sequence truncation.

In [ ]:
def preprocess(batch):
    inputs = tokenizer(batch['input_text'], max_length=256, truncation=True, padding="max_length")
    labels = tokenizer(text_target=batch['target_text'], max_length=48, truncation=True, padding="max_length")
    labels_ids = labels["input_ids"]
    labels_ids = [[(t if t != tokenizer.pad_token_id else -100) for t in seq] for seq in labels_ids]
    inputs["labels"] = labels_ids
    return inputs

tokenized_train = Dataset.from_pandas(train_dual).map(
    preprocess, 
    batched=True, 
    batch_size=256, 
    remove_columns=train_dual.columns.tolist()
)

print(f"Tokenization complete: {len(tokenized_train)} training instances.")

# Model Fine-Tuning
Runthe sequence-to-sequence training loop.

In [ ]:
training_args = Seq2SeqTrainingArguments(
    output_dir="./mt5_runs",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=3e-4,
    num_train_epochs=1,
    weight_decay=0.01,
    logging_steps=100,
    eval_strategy="no",
    save_strategy="no",
    fp16=False,
    dataloader_num_workers=0,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    processing_class=tokenizer,
)

print("Starting training...")
t0 = time.time()
trainer.train()
print(f"Training completed successfully in {(time.time() - t0)/60:.2f} minutes!")

# Inference, Post-Processing & CSV Submission
Run batch inference with task-specific decoding, cleans any leading topic tags from headlines, and saves submission.csv.

In [ ]:
model.eval()
ALLOWED_TOPICS = {'business', 'health', 'politics', 'religion', 'sports', 'entertainment', 'technology'}
test_predictions = []

test_topics = test_dual[test_dual['task'] == 'topic'].copy().reset_index(drop=True)
test_headlines = test_dual[test_dual['task'] == 'headline'].copy().reset_index(drop=True)

# 1. Topic generation (Greedy)
print("Generating Task A topics...")
for i in tqdm(range(0, len(test_topics), 32)):
    batch = test_topics.iloc[i : i + 32]
    inputs = tokenizer(batch['input_text'].tolist(), max_length=256, truncation=True, padding=True, return_tensors="pt").to(device)
    with torch.no_grad():
        ids = model.generate(**inputs, max_new_tokens=6, num_beams=1)
    decoded = tokenizer.batch_decode(ids, skip_special_tokens=True)
    for row_id, pred in zip(batch['id'], decoded):
        clean = pred.strip().lower()
        matched = [t for t in ALLOWED_TOPICS if t in clean]
        test_predictions.append({'id': row_id, 'prediction': matched[0] if matched else 'politics'})

# 2. Headline generation (Beams + cleanup)
print("Generating Task B headlines...")
for i in tqdm(range(0, len(test_headlines), 16)):
    batch = test_headlines.iloc[i : i + 16]
    inputs = tokenizer(batch['input_text'].tolist(), max_length=256, truncation=True, padding=True, return_tensors="pt").to(device)
    with torch.no_grad():
        ids = model.generate(**inputs, max_new_tokens=45, min_new_tokens=8, num_beams=3, repetition_penalty=1.2, early_stopping=True)
    decoded = tokenizer.batch_decode(ids, skip_special_tokens=True)
    for row_id, pred, orig_text in zip(batch['id'], decoded, test_df['text'].iloc[i : i + 16]):
        clean = pred.strip()
        clean = re.sub(r'^(politics|sports|business|entertainment|health|religion|technology)\s*[:\-–]\s*', '', clean, flags=re.IGNORECASE).strip()
        if clean.lower() in ALLOWED_TOPICS or len(clean.split()) < 3:
            clean = str(orig_text).strip().split('.')[0][:80]
        test_predictions.append({'id': row_id, 'prediction': clean})

# 3. Align and write submission
sub_df = pd.DataFrame(test_predictions)
order_map = {row_id: idx for idx, row_id in enumerate(test_dual['id'])}
sub_df['sort_key'] = sub_df['id'].map(order_map)
sub_df = sub_df.sort_values('sort_key').drop(columns=['sort_key']).reset_index(drop=True)

sub_df.to_csv('/kaggle/working/submission.csv', index=False)
sub_df.to_csv('submission.csv', index=False)

print("\n--- FINAL VERIFICATION ---")
print(f"Total Rows : {len(sub_df)} (Target: 3486)")
print(f"Nulls      : {sub_df.isnull().sum().sum()}")
print("\nFirst 6 rows of submission.csv:")
print(sub_df.head(6))